In [3]:
# ============================================================================
#  ANONYMISATION TAG ABLATION — SINGLE COLAB CELL
#  Run on GPU T4 runtime. Paste everything into one cell and execute.
# ============================================================================

# 1. Monta o Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install dependencies
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.48.3", "scikit-learn==1.6.1",
                "accelerate>=1.3.0", "sentencepiece", "sacremoses"], check=True)
print("Dependencies installed.\n")

# 3. Direct project path (Models_BERT is at the Drive root)
import os, glob, re, json
import numpy as np
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive"
print("PROJECT_ROOT =", PROJECT_ROOT)

BASE      = Path(PROJECT_ROOT)
exp_hits  = sorted(glob.glob(str(BASE / "Models_BERT/experiments/bertimbau_large__*/best_model")))
assert exp_hits, f"best_model not found in {BASE}/Models_BERT/experiments/ — check PROJECT_ROOT"
MODEL_DIR = Path(exp_hits[0])
TEST_FILE = BASE / "Models_BERT/Data/processed/test.jsonl"
OUT_FILE  = MODEL_DIR.parent / "ablation_tags_result.json"
print("MODEL_DIR =", MODEL_DIR)
print("TEST_FILE =", TEST_FILE)
assert TEST_FILE.exists(), f"test.jsonl not found: {TEST_FILE}"

# 4. Load test set and create tag-stripped version
TAG_PATTERN = re.compile(
    r'\[(?:CPF|CNPJ|EMAIL|TELEFONE|CEP|OAB|VALOR|URL|PROCESSO|PESSOA)\]')

def strip_tags(text):
    return re.sub(r'\s+', ' ', TAG_PATTERN.sub('', text)).strip()

records = [json.loads(l) for l in open(TEST_FILE, encoding='utf-8')]
texts_orig  = [r['text'] for r in records]
texts_clean = [strip_tags(r['text']) for r in records]
labels      = [r['label'] for r in records]
CLASS_NAMES = ['tese', 'fato', 'ruido']

n_with_tags = sum(1 for t in texts_orig if TAG_PATTERN.search(t))
print(f"\nTest set: {len(records)} samples")
print(f"With at least one tag: {n_with_tags} ({100*n_with_tags/len(records):.1f}%)")

# 5. Carrega modelo treinado
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR)).eval().to(device)

# 6. Batch inference
def predict(texts, batch_size=64):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, max_length=512, truncation=True,
                        padding=True, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**enc).logits.cpu().numpy()
        preds.extend(logits.argmax(axis=1).tolist())
        print(f"  {min(i+batch_size, len(texts))}/{len(texts)}", end='\r')
    print()
    return np.array(preds)

print("\n--- With tags (reproduction) ---")
preds_orig  = predict(texts_orig)
print("--- Without tags (ablation) ---")
preds_clean = predict(texts_clean)

# 7. Metrics
from sklearn.metrics import f1_score, classification_report
y_true = np.array(labels)

f1_orig  = f1_score(y_true, preds_orig,  average='macro')
f1_clean = f1_score(y_true, preds_clean, average='macro')
rep_orig  = classification_report(y_true, preds_orig,  target_names=CLASS_NAMES, output_dict=True)
rep_clean = classification_report(y_true, preds_clean, target_names=CLASS_NAMES, output_dict=True)

print("\n========================================")
print(f"F1 macro WITH tags:    {f1_orig:.4f}")
print(f"F1 macro WITHOUT tags: {f1_clean:.4f}")
print(f"Delta:                {f1_clean - f1_orig:+.4f}")
print()
print(f"{'Class':<10} {'With tags':>10} {'Without tags':>12} {'Delta':>8}")
for cls in CLASS_NAMES:
    w  = rep_orig[cls]['f1-score']; wo = rep_clean[cls]['f1-score']
    print(f"{cls:<10} {w:>10.4f} {wo:>10.4f} {wo-w:>+8.4f}")
print(f"\nErrors WITH tags:    {int((preds_orig  != y_true).sum())}")
print(f"Errors WITHOUT tags: {int((preds_clean != y_true).sum())}")
print("========================================")

# 8. Save result to Drive
result = {
    "f1_macro_with_tags":    round(f1_orig, 4),
    "f1_macro_without_tags": round(f1_clean, 4),
    "delta_macro":           round(f1_clean - f1_orig, 4),
    "errors_with_tags":      int((preds_orig  != y_true).sum()),
    "errors_without_tags":   int((preds_clean != y_true).sum()),
    "samples_with_tags_pct": round(100*n_with_tags/len(records), 1),
    "per_class": {
        cls: {"f1_with":    round(rep_orig[cls]['f1-score'],  4),
              "f1_without": round(rep_clean[cls]['f1-score'], 4),
              "delta":      round(rep_clean[cls]['f1-score'] - rep_orig[cls]['f1-score'], 4)}
        for cls in CLASS_NAMES}
}
OUT_FILE.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
print(f"\nResult saved to: {OUT_FILE}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dependências instaladas.

PROJECT_ROOT = /content/drive/MyDrive
MODEL_DIR = /content/drive/MyDrive/Models_BERT/experiments/bertimbau_large__20260602_181208__e9f4e659/best_model
TEST_FILE = /content/drive/MyDrive/Models_BERT/Data/processed/test.jsonl

Test set: 2559 amostras
Com ao menos 1 tag: 2338 (91.4%)
Device: cuda

--- Com tags (reprodução) ---
  2559/2559
--- Sem tags (ablação) ---
  2559/2559

F1 macro COM tags:    0.9901
F1 macro SEM tags:    0.9913
Delta:                +0.0012

Classe       Com tags   Sem tags    Delta
tese           0.9847     0.9865  +0.0018
fato           0.9863     0.9880  +0.0017
ruido          0.9994     0.9994  +0.0000

Erros COM tags:    25
Erros SEM tags:    22

Resultado salvo em: /content/drive/MyDrive/Models_BERT/experiments/bertimbau_large__20260602_181208__e9f4e659/ablation_tags_result.json
